# Declino italiano: dashboard API

Cruscotto comparabile Italia-Germania-Francia-Spagna-UE costruito solo da API ufficiali.
Le fonti sono World Bank (WDI, WGI, Doing Business storico) ed Eurostat (`nama_10_lp_ulc`).

L'obiettivo non e produrre uno slogan, ma rendere riproducibile la diagnosi: per ogni indicatore guardiamo trend dell'Italia e gap rispetto ai principali peer europei.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "analisi").exists():
    ROOT = ROOT.parent
if not (ROOT / "analisi").exists():
    raise RuntimeError("Esegui il notebook dalla root del repo o da una sottocartella del repo.")
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display

from analisi.utils.api_indicators import run_decline_analysis, plot_indicator

# I dati vengono richiesti alle API ufficiali a ogni esecuzione.
REFRESH_API = True

result = run_decline_analysis(refresh=REFRESH_API)
panel = result["panel"]
summary = result["summary"]
transforms = result["transforms"]

summary.head(10)


## Come leggere la tabella

`bad_when` e la direzione interpretativa: `low` significa che valori piu bassi sono peggiori, `high` che valori piu alti sono peggiori.  
`worsened_since_baseline` segnala peggioramento rispetto al primo dato disponibile dal 2000.  
`italy_worse_than_peer_avg` segnala Italia peggiore della media Germania-Francia-Spagna-UE nell'ultimo anno disponibile.


In [ ]:
cols = [
    "theme", "indicator_label", "unit", "latest_year", "latest_value_italy",
    "change_abs_since_baseline", "gap_vs_peer_mean",
    "worsened_since_baseline", "italy_worse_than_peer_avg", "source"
]
critical = summary[
    summary["worsened_since_baseline"] | summary["italy_worse_than_peer_avg"]
].sort_values(["theme", "italy_worse_than_peer_avg", "worsened_since_baseline"], ascending=[True, False, False])
critical[cols]


## Quadro per tema

Qui contiamo quanti indicatori, dentro ogni tema, segnalano peggioramento storico o svantaggio rispetto ai peer.


In [ ]:
by_theme = (
    summary.assign(criticita=summary["worsened_since_baseline"] | summary["italy_worse_than_peer_avg"])
    .groupby("theme", as_index=False)
    .agg(indicatori=("indicator_id", "nunique"), indicatori_critici=("criticita", "sum"))
    .sort_values("indicatori_critici", ascending=False)
)
by_theme


## Grafici chiave

Le linee mostrano Italia e peer. L'Italia e evidenziata con tratto piu spesso.


In [ ]:
key_indicators = [
    "ESTAT_RLPR_HW_I15",
    "SP.DYN.TFRT.IN",
    "SP.POP.65UP.TO.ZS",
    "GB.XPD.RSDV.GD.ZS",
    "PAY.TAX.TM",
    "GC.XPN.INTP.RV.ZS",
    "NE.GDI.FTOT.ZS",
    "GOV_WGI_GE.EST",
]

for indicator_id in key_indicators:
    display(Image(filename=str(plot_indicator(panel, indicator_id))))


## Tracciabilita fonti

Ogni riga conserva `source_url`, cioe l'endpoint API o la combinazione di endpoint usata per gli indicatori derivati.


In [ ]:
summary[["indicator_id", "indicator_label", "source", "source_url"]].drop_duplicates().sort_values("indicator_label")
